In [6]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [7]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

'08'

In [12]:
template_file = 'report_for_BD_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

'report_for_BD_2026-09-22.xlsx'

In [ ]:
# 1. Daily Badge impression by device     (no need)
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2

In [ ]:
# 2. Daily Brand Page (LMS) Pageview         (no need? = may be is sr1 page view?)
# 進入search頁面後，點擊少鹽少糖食店按鈕
# 17:35:05|| or.search.layer.search| CityID:0;geo:22.2915336%2C114.2081752;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# unique users; pageview: total click count
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = """
SELECT
    date(time) as querydate,
    platform,
    count(1) as count
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE LOWER(EventAction) LIKE '%or.search.layer.search%'
    AND LOWER(EventLabelRaw) LIKE '%sr:lmssr1%'
GROUP BY 1, 2
"""

In [13]:
#少鹽少糖食店SR1 page view

sql = f'''
with sr1 as
  (select '$$$少鹽少糖食店$$$' AS dummy, platform
  from `openrice-production.ORGA.PV_{year}{str_month}*`
  where -- eventcategory in ('Search Related', 'WebEvent')
   (lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'dedicatedpromotionid')) like '13' or 
       lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'amtid')) like '1093' or
       lower(eventdata) like '%hongkong%amenityid=1093%'))

select platform, count(1) from sr1
group by platform
    '''

df_big_query = client.query(sql).result().to_dataframe()

web = df_big_query.query("platform=='mobile' | platform=='desktop'  ").f0_.sum()
app = df_big_query.query("platform=='android' | platform=='ios' | platform=='hms' ").f0_.sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=2, header=None, index=False)

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [14]:
#少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in  (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=7, header=None, index=False)
    

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenalee\AppData\Local\Temp\ipykernel_8656\4189330246.py:9: SyntaxWarning: invalid escape sequence '\d'
  '''
C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# 3. Daily Theme Listing Impressions --> adv search
# 進入少鹽少糖食店頁面後，點擊篩選搜尋按鈕
# 09:23:47|| or.advsearch.open.filter| CityID:0;sr:qcksearch;Lang:zh_TW;Ver:7.20.4; sn:hkhk.LMS2.35336.tab.-990.1
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = f"""
with adv_search as (
    select '$$$少鹽少糖食店$$$' AS dummy, platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'or.advsearch.open.filter'
    AND LOWER(EventLabelRaw) LIKE '%hk.lms2.%')
)

select platform, count(1) as count
from adv_search
group by platform
"""

df_big_query_3 = client.query(sql).result().to_dataframe()

web = df_big_query_3.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
app = df_big_query_3.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
